## TAHAP 0 — Import Libraries

In [31]:
# !pip install pandas numpy scikit-learn matplotlib datasets pyarrow
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 50)
print("Setup selesai.")

Setup selesai.


## TAHAP 1 — Load Dataset (Amazon Reviews 2023)

Dataset = 2 file per kategori: `review` (interaksi user) + `meta` (info produk).
Pilih **1 kategori kecil** (contoh: `All_Beauty`).

> Opsi BigQuery: kamu bisa juga meng-upload kedua file ke BigQuery, JOIN + cleaning awal
> dengan SQL, lalu tarik ke Python pakai `pandas_gbq.read_gbq(...)`. Setelah itu alurnya sama.

In [32]:
import urllib.request, os

CATEGORY = "Video_Games"
url_review = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/{CATEGORY}.jsonl.gz"
url_meta   = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_{CATEGORY}.jsonl.gz"

f_review = f"{CATEGORY}_review.jsonl.gz"
f_meta   = f"meta_{CATEGORY}.jsonl.gz"

# Download ke file dulu (hanya kalau belum ada -> tidak download ulang)
if not os.path.exists(f_review):
    print("Downloading review...")
    urllib.request.urlretrieve(url_review, f_review)
if not os.path.exists(f_meta):
    print("Downloading meta...")
    urllib.request.urlretrieve(url_meta, f_meta)

print("Ukuran file:")
print(f"  review: {os.path.getsize(f_review)/1e6:.1f} MB")
print(f"  meta  : {os.path.getsize(f_meta)/1e6:.1f} MB")

Ukuran file:
  review: 814.2 MB
  meta  : 103.1 MB


In [33]:
import pandas as pd

# Baca dari file yang sudah terunduh
reviews = pd.read_json("Video_Games_review.jsonl.gz", lines=True, compression="gzip")
meta    = pd.read_json("meta_Video_Games.jsonl.gz",   lines=True, compression="gzip")

print("reviews:", reviews.shape, "| meta:", meta.shape)

# Cek densitas & kualitas teks (untuk keputusan fitur)
user_counts = reviews['user_id'].value_counts()
item_counts = reviews['parent_asin'].value_counts()
print(f"\nUser dengan >=5 review : {(user_counts >= 5).sum()}")
print(f"Produk dengan >=5 review: {(item_counts >= 5).sum()}")

print(f"\n'description' kosong: {meta['description'].apply(lambda x: len(str(x)) < 5).mean():.1%}")
print(f"'features' kosong   : {meta['features'].apply(lambda x: len(str(x)) < 5).mean():.1%}")
print(f"'title' kosong      : {meta['title'].apply(lambda x: len(str(x)) < 5).mean():.1%}")

reviews: (4624615, 10) | meta: (137269, 16)

User dengan >=5 review : 117742
Produk dengan >=5 review: 63180

'description' kosong: 37.7%
'features' kosong   : 28.8%
'title' kosong      : 0.1%


In [34]:
from google.cloud import bigquery

PROJECT_ID = "amazon-recsys-499804"
client = bigquery.Client(project=PROJECT_ID)
print("Terhubung ke:", PROJECT_ID)

# Tes koneksi
datasets = list(client.list_datasets())
print(f"Jumlah dataset saat ini: {len(datasets)}")
print("Koneksi BigQuery BERHASIL")

Python(9118) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Terhubung ke: amazon-recsys-499804
Jumlah dataset saat ini: 1
Koneksi BigQuery BERHASIL


## Buat dataset (wadah tabel) di BigQuery

In [35]:
DATASET_ID = "amazon_recsys"

dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset.location = "US"
client.create_dataset(dataset, exists_ok=True)
print(f"Dataset '{DATASET_ID}' siap")

Dataset 'amazon_recsys' siap


## Rapikan & flatten data (Transform)

In [36]:
import pandas as pd

def flatten(x):
    if isinstance(x, (list, tuple)):
        return " ".join(map(str, x))
    return str(x) if pd.notna(x) else ""

# META — pilih kolom + flatten yang berbentuk list
meta_clean = pd.DataFrame({
    "parent_asin": meta["parent_asin"].astype(str),
    "product_title": meta["title"].fillna("").astype(str),
    "description": meta["description"].apply(flatten),
    "features": meta["features"].apply(flatten),
    "categories": meta["categories"].apply(flatten),
    "price": pd.to_numeric(meta["price"], errors="coerce"),
    "average_rating": pd.to_numeric(meta["average_rating"], errors="coerce"),
    "rating_number": pd.to_numeric(meta["rating_number"], errors="coerce"),
})

# REVIEWS — pilih kolom inti
reviews_clean = pd.DataFrame({
    "user_id": reviews["user_id"].astype(str),
    "parent_asin": reviews["parent_asin"].astype(str),
    "rating": pd.to_numeric(reviews["rating"], errors="coerce"),
    "timestamp": reviews["timestamp"],
})

print("meta_clean:", meta_clean.shape, "| reviews_clean:", reviews_clean.shape)
meta_clean.head(2)

meta_clean: (137269, 8) | reviews_clean: (4624615, 4)


,parent_asin,product_title,description,features,categories,price,average_rating,rating_number
0,B000FH0MHO,Dash 8-300 Professional Add-On,The Dash 8-300 Professional Add-On lets you pi...,Features Dash 8-300 and 8-Q300 ('Q' rollout li...,Video Games PC Games,NaN,5.0,1
1,B00069EVOG,Phantasmagoria: A Puzzle of Flesh,,Windows 95,Video Games PC Games,NaN,4.1,18


## Upload ke BigQuery (Load)

In [37]:
# WRITE_TRUNCATE membuat upload idempoten:
# tabel ditimpa setiap run, jadi tidak ada duplikat walau di-restart/run berkali-kali
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

In [38]:
from google.cloud import bigquery

# WRITE_TRUNCATE = hapus isi tabel dulu, tulis baru -> bersih dari duplikat
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

# Upload meta (timpa)
job = client.load_table_from_dataframe(meta_clean, f"{DATASET_ID}.meta", job_config=job_config)
job.result()
print("✅ meta:", client.get_table(f"{DATASET_ID}.meta").num_rows, "baris")

# Upload reviews (timpa)
job = client.load_table_from_dataframe(reviews_clean, f"{DATASET_ID}.reviews", job_config=job_config)
job.result()
print("✅ reviews:", client.get_table(f"{DATASET_ID}.reviews").num_rows, "baris")

✅ meta: 137269 baris
✅ reviews: 4624615 baris


#### Cek 1 — Tabel apa saja yang ada di dataset

In [39]:
tables = client.list_tables(f"{PROJECT_ID}.{DATASET_ID}")
print("Tabel yang ada di dataset:")
for t in tables:
    print(" -", t.table_id)

Tabel yang ada di dataset:
 - meta
 - reviews


#### Cek 2 — Kolom & tipe data tiap tabel (skema)

In [40]:
# Skema tabel reviews
print("=== SKEMA: reviews ===")
table_reviews = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.reviews")
for field in table_reviews.schema:
    print(f"  {field.name:20s} {field.field_type}")
print(f"  Total baris: {table_reviews.num_rows}\n")

# Skema tabel meta
print("=== SKEMA: meta ===")
table_meta = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.meta")
for field in table_meta.schema:
    print(f"  {field.name:20s} {field.field_type}")
print(f"  Total baris: {table_meta.num_rows}")

=== SKEMA: reviews ===
  user_id              STRING
  parent_asin          STRING
  rating               INTEGER
  timestamp            DATETIME
  Total baris: 4624615

=== SKEMA: meta ===
  parent_asin          STRING
  product_title        STRING
  description          STRING
  features             STRING
  categories           STRING
  price                FLOAT
  average_rating       FLOAT
  rating_number        INTEGER
  Total baris: 137269


#### Cek 3 — Lihat isi sampel (beberapa baris)

In [41]:
# Preview 5 baris reviews
print("=== SAMPLE: reviews ===")
sample_reviews = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.reviews` LIMIT 5
""").to_dataframe()
display(sample_reviews)

# Preview 5 baris meta
print("=== SAMPLE: meta ===")
sample_meta = client.query(f"""
    SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.meta` LIMIT 5
""").to_dataframe()
display(sample_meta)

=== SAMPLE: reviews ===


/opt/homebrew/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_id,parent_asin,rating,timestamp
0,AG6BAEKWLCWH2TW3KKLVK773YF6A,B017V6YVDC,1,2020-12-12 00:54:34.794
1,AG6BAEKWLCWH2TW3KKLVK773YF6A,B0B36HX334,1,2018-04-09 22:45:03.402
2,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B08SM7T6FF,1,2021-04-24 01:27:16.331
3,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,B08R5B7YS4,1,2021-01-24 03:41:06.223
4,AEYFU5FQAJYXFREC4KHQ34EGZ3LQ,B07GJ5W7HV,1,2019-07-03 18:15:42.305


=== SAMPLE: meta ===


/opt/homebrew/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,parent_asin,product_title,description,features,categories,price,average_rating,rating_number
0,B00E1EBX4I,Green Aluminum Coated Black Hard Case Cover fo...,,,,NaN,1.0,1
1,B07ZJ97BPV,USA Version classic edtion super games 68 in 1...,USA Version classic edtion super games 68 in 1...,68 in 1 SNES Game Cartridge 16 Bit SNES Games ...,,NaN,1.0,2
2,B00D8YX2OG,"Farming, Agriculture and Woodcutting Simulator...",,,,NaN,1.0,1
3,B00851S9JU,Curling 2012,,,,NaN,1.0,3
4,B00SYZDPA0,ModFreakz™ Console/Controller Vinyl Skin Set -...,,,,NaN,1.0,1


## query join + k-core

"K-core filtering menyaring user dan produk yang punya minimal k interaksi (saya pakai k=5). 
Tujuannya mengurangi sparsity data membuang user yang riwayatnya terlalu tipis untuk diprofilkan dan produk yang datanya terlalu sedikit untuk diandalkan. Hasilnya data lebih padat dan model lebih berkualitas."

In [42]:
query = f"""
WITH user_ok AS (
    SELECT user_id
    FROM `{PROJECT_ID}.{DATASET_ID}.reviews`
    GROUP BY user_id HAVING COUNT(*) >= 5
),
item_ok AS (
    SELECT parent_asin
    FROM `{PROJECT_ID}.{DATASET_ID}.reviews`
    GROUP BY parent_asin HAVING COUNT(*) >= 5
)
SELECT r.user_id, r.parent_asin, r.rating, r.timestamp,
       m.product_title, m.description, m.features, m.categories, m.price
FROM `{PROJECT_ID}.{DATASET_ID}.reviews` r
JOIN `{PROJECT_ID}.{DATASET_ID}.meta` m USING (parent_asin)
WHERE r.user_id IN (SELECT user_id FROM user_ok)
  AND r.parent_asin IN (SELECT parent_asin FROM item_ok)
"""

df = client.query(query).to_dataframe()
print("Hasil setelah join + k-core:", df.shape)
df.head()

/opt/homebrew/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Hasil setelah join + k-core: (994883, 9)


,user_id,parent_asin,rating,timestamp,product_title,description,features,categories,price
0,AHC4ZAH4OD7K2UQ5QONDOAAXINWQ,B004MPR0ZC,5,2015-02-09 12:57:25.000,CTA Digital Nintendo 3Ds Cartridge Storage Sol...,Product Description The Nintendo 3DS is the fi...,"Stores & protects 22 3DS game cartridges, 2 re...",Video Games Legacy Systems Nintendo Systems Ni...,NaN
1,AGVY6H5SMX5LNR4GJJP5TUWYMHDQ,B01AO98EXQ,5,2019-06-08 21:40:43.270,HDE Data and Power Cable for PSP Go Portable S...,,,Video Games Legacy Systems PlayStation Systems...,NaN
2,AFZKBGDK67VIKTN4O3HLSF6AUP5A,B001FVQO3U,5,2012-02-06 18:46:29.000,Dream Day Wedding Destinations - Nintendo DS,Product Description One of the most successful...,Love is in the Air - Dream Day Wedding Destina...,Video Games Legacy Systems Nintendo Systems,24.97
3,AFVPC4Z3O3V76COIGC3L33NWY6YA,B000MIXFWA,5,2007-03-19 19:37:40.000,PS3 Component AV Cable,Sony 98044 Playstation3 Component Av Cable,Features Separate Left And Right Audio Plugs F...,Video Games Legacy Systems PlayStation Systems...,69.97
4,AHPQXKJKE2SI4KC5QMYG3WE7VQCA,B00NXKBTP4,5,2015-06-16 18:15:40.000,"Skylanders Trap Team Master Knight Mare, Figur",Before the destruction of the Core of Light le...,Knight Mare Nowhere to Hide,Video Games Legacy Systems PlayStation Systems...,54.50


In [43]:
# Simpan hasil query ke file lokal -> tidak perlu query BigQuery ulang
df.to_parquet("video_games_kcore.parquet")
print("Tersimpan! Ukuran:", df.shape)

Tersimpan! Ukuran: (994883, 9)


In [44]:
print("Total baris (interaksi):", len(df))
print("Produk unik (item):", df['parent_asin'].nunique())
print("User unik:", df['user_id'].nunique())

Total baris (interaksi): 994883
Produk unik (item): 55453
User unik: 117727


## TAHAP 2 — Preprocessing

Join + cleaning: tangani missing value, duplikat, tipe data, dan **flatten `categories`**
(yang berbentuk list) menjadi string dipisah koma — ini yang nanti di-tokenisasi
(analog `genre`).

In [45]:
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== DUPLIKAT ===")
print("Baris duplikat penuh:", df.duplicated().sum())
print("Duplikat (user+produk):", df.duplicated(subset=["user_id","parent_asin"]).sum())

print("\n=== RATING (harus 1-5) ===")
print(df["rating"].describe())
print("Rating di luar 1-5:", ((df["rating"]<1)|(df["rating"]>5)).sum())

print("\n=== HARGA ===")
print("Harga negatif:", (df["price"]<0).sum())
print("Harga NaN:", df["price"].isna().sum(), f"({df['price'].isna().mean()*100:.1f}%)")

print("\n=== TEKS untuk fitur ===")
print("product_title kosong:", (df["product_title"].fillna("").str.len()<2).sum())
print("description kosong   :", (df["description"].fillna("").str.len()<2).sum())

=== MISSING VALUES ===
user_id               0
parent_asin           0
rating                0
timestamp             0
product_title         0
description           0
features              0
categories            0
price            265845
dtype: int64

=== DUPLIKAT ===
Baris duplikat penuh: 19879
Duplikat (user+produk): 29989

=== RATING (harus 1-5) ===
count    994883.0
mean     4.247629
std      1.213787
min           1.0
25%           4.0
50%           5.0
75%           5.0
max           5.0
Name: rating, dtype: Float64
Rating di luar 1-5: 0

=== HARGA ===
Harga negatif: 0
Harga NaN: 265845 (26.7%)

=== TEKS untuk fitur ===
product_title kosong: 61
description kosong   : 194094


In [46]:
print("Sebelum cleaning:", df.shape)

# 1. Buang baris duplikat penuh
df_clean = df.drop_duplicates().copy()
print("Setelah buang duplikat penuh:", df_clean.shape)

# 2. Tangani duplikat user-produk: simpan rating TERAKHIR (paling baru)
#    Kenapa terakhir? Rating terbaru paling mencerminkan preferensi terkini user.
df_clean = (df_clean
            .sort_values("timestamp")
            .drop_duplicates(subset=["user_id", "parent_asin"], keep="last")
            .reset_index(drop=True))
print("Setelah buang duplikat user-produk:", df_clean.shape)

# 3. Buang produk tanpa judul (61 baris) — tidak bisa dibuat fitur/ditampilkan
df_clean = df_clean[df_clean["product_title"].fillna("").str.len() >= 2].reset_index(drop=True)
print("Setelah buang judul kosong:", df_clean.shape)

# Ringkasan akhir
print("\n=== SETELAH CLEANING ===")
print("Interaksi :", len(df_clean))
print("User unik :", df_clean["user_id"].nunique())
print("Produk    :", df_clean["parent_asin"].nunique())

Sebelum cleaning: (994883, 9)
Setelah buang duplikat penuh: (975004, 9)
Setelah buang duplikat user-produk: (964894, 9)
Setelah buang judul kosong: (964834, 9)

=== SETELAH CLEANING ===
Interaksi : 964834
User unik : 117727
Produk    : 55450


In [47]:
df_clean.to_parquet("video_games_clean.parquet")
print("Data bersih tersimpan:", df_clean.shape)

Data bersih tersimpan: (964834, 9)
